In [58]:
# Cell 1: Imports and Setup
import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
from copy import deepcopy
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('scaling_notebook')


In [59]:

# Cell 2: Add Project Path
current_dir = os.getcwd()
if not any(p.endswith('dendritic_modeling') for p in sys.path):
    if os.path.basename(current_dir) == 'notebooks':
        project_root = os.path.dirname(current_dir)
    else:
        project_root = current_dir
    sys.path.insert(0, project_root)
    logger.info(f"Added {project_root} to Python path")



2025-02-25 13:28:29,228 - scaling_notebook - INFO - Added /n/holylabs/LABS/kempner_dev/Users/hsafaai/Code/dendritic-modeling to Python path


In [60]:
# Cell 3: Import Project Modules
from dendritic_modeling.config import load_config
from dendritic_modeling.models import ProbabilisticClassifier, Classifier
from dendritic_modeling.networks import ExcitationInhibitionNetwork, MLPExcInhNetwork
from dendritic_modeling.synthetic_datasets import get_unified_datasets



In [61]:
# Cell 4: Configuration Preparation Function
def load_config_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def prepare_config(base_config_path, dim, config_variations):
    config = load_config(base_config_path)
    
    # Network type and dimension
    net_type = config_variations.get('net_type', config.model.network.type)
    config.model.network.type = net_type
    
    # Set dimensions and specific config variations
    if net_type == "EINet":
        config.model.network.parameters.excitatory_branch_factors = [2, int(dim)]
        config.model.network.parameters.inhibitory_branch_factors = []
        
        # Apply specific EINet variations
        for key, value in config_variations.items():
            if key.startswith('einet_'):
                param_name = key[6:]
                setattr(config.model.network.parameters, param_name, value)
    
    elif net_type == "MLP":
        config.model.network.parameters.hidden_layer_sizes = [int(dim), int(dim//2)]
    
    return config



In [62]:
# Cell 5: Model Initialization Function
def initialize_model(model_cfg):
    task = model_cfg.task
    probabilistic = model_cfg.probabilistic
    net_type = model_cfg.network.type
    net_params = model_cfg.network.parameters.__dict__
    
    if task == 'classification':
        if net_type == 'MLP':
            logger.info(f"Creating MLP network")
            net = MLPExcInhNetwork(**net_params)
            output_dim = net_params.get('output_dim', 10)
            if probabilistic:
                return ProbabilisticClassifier(net, output_dim)
            else:
                return Classifier(net)
        
        elif net_type == 'EINet':
            logger.info(f"Creating EINet network")
            net = ExcitationInhibitionNetwork(**net_params)
            output_dim = net_params['excitatory_layer_sizes'][-1]
            if probabilistic:
                return ProbabilisticClassifier(net, output_dim)
            else:
                return Classifier(net)
        
        else:
            raise ValueError(f"Invalid network type: {net_type}")
    else:
        raise ValueError(f"Invalid task: {task}")



In [63]:
# Cell 6: Model Evaluation Function
def evaluate_model(model, test_loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            if hasattr(model, 'compute_loss'):
                loss = model.compute_loss(inputs, targets)
                total_loss += loss.item()
            
            if hasattr(model, 'predict'):
                predictions = model.predict(inputs)
                correct += (predictions == targets).sum().item()
            else:
                outputs = model(inputs)
                predictions = outputs.argmax(dim=1)
                correct += (predictions == targets).sum().item()
            
            total += targets.size(0)
    
    accuracy = correct / total if total > 0 else 0
    avg_loss = total_loss / len(test_loader) if len(test_loader) > 0 else 0
    
    return accuracy, avg_loss



In [64]:
# Cell 7: Experiment Running Functions
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def run_single_experiment(base_config_path, dim, config_variations, epochs=5):
    config = prepare_config(base_config_path, dim, config_variations)
    
    train_ds, valid_ds, test_ds = get_unified_datasets(config.task, config.train)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64)
    
    model = initialize_model(config.model)
    
    param_count = count_parameters(model)
    logger.info(f"{config_variations.get('net_type', 'Network')} config={config_variations}: Parameter count = {param_count}")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    accuracy, loss = evaluate_model(model, test_loader, device)
    logger.info(f"Config {config_variations}: Accuracy = {accuracy:.4f}, Loss = {loss:.4f}")
    
    return {
        "config": config_variations,
        "param_count": param_count, 
        "accuracy": accuracy, 
        "loss": loss
    }

def run_scaling_experiment(base_config_path, dim_list, experiment_configs, epochs=5):
    results = {}
    
    for config_name, config_variations in experiment_configs.items():
        results[config_name] = []
        
        for dim in dim_list:
            variations = deepcopy(config_variations)
            variations['dim'] = dim
            result = run_single_experiment(base_config_path, dim, variations, epochs)
            results[config_name].append(result)
    
    return results



In [65]:
# Cell 8: Plotting Function
def plot_scaling_laws(results):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax2.set_xscale("log")
    ax2.set_yscale("log")
    
    styles = {
        "EINet_shunting": {"color": "blue", "marker": "o"},
        "EINet_no_shunting": {"color": "red", "marker": "s"},
        "MLP": {"color": "green", "marker": "^"}
    }
    
    for config_name, data_list in results.items():
        data_list = sorted(data_list, key=lambda x: x["param_count"])
        
        x_param = [d["param_count"] for d in data_list]
        y_loss = [max(1e-10, d["loss"]) for d in data_list]
        y_error = [max(1e-10, 1.0 - d["accuracy"]) for d in data_list]
        dims = [d.get("config", {}).get("dim", "N/A") for d in data_list]
        
        style = styles.get(config_name, {"color": "purple", "marker": "x"})
        
        ax1.plot(x_param, y_loss, marker=style["marker"], color=style["color"], 
                 label=f"{config_name} (Loss)", linewidth=2, markersize=10)
        
        ax2.plot(x_param, y_error, marker=style["marker"], color=style["color"], 
                 label=f"{config_name} (Error)", linewidth=2, markersize=10)
        
        for x, y1, y2, dim in zip(x_param, y_loss, y_error, dims):
            ax1.annotate(f"dim={dim}", xy=(x, y1), xytext=(10, 0),
                        textcoords='offset points', fontsize=10)
            ax2.annotate(f"dim={dim}", xy=(x, y2), xytext=(10, 0),
                        textcoords='offset points', fontsize=10)
    
    ax1.set_xlabel("Parameter Count (log scale)", fontsize=12)
    ax1.set_ylabel("Loss (log scale)", fontsize=12)
    ax1.set_title("Loss vs Parameter Count", fontsize=14)
    ax1.grid(True, which="both", alpha=0.3)
    ax1.legend(loc='best', fontsize=12)
    
    ax2.set_xlabel("Parameter Count (log scale)", fontsize=12)
    ax2.set_ylabel("Error Rate (log scale)", fontsize=12)
    ax2.set_title("Error Rate vs Parameter Count", fontsize=14)
    ax2.grid(True, which="both", alpha=0.3)
    ax2.legend(loc='best', fontsize=12)
    
    plt.tight_layout()
    return fig



In [66]:
# Cell 9: Main Execution
if __name__ == "__main__":
    BASE_CONFIG = "../.vscode/config_exp.yaml"
    DIM_LIST = [2, 4, 8, 16, 32]
    EPOCHS = 5
    
    experiment_configs = {
        "EINet_shunting": {
            "net_type": "EINet",
            "einet_use_shunting": True,
            "einet_reactivation_type": "param_tanh"
        },
        "EINet_no_shunting": {
            "net_type": "EINet", 
            "einet_use_shunting": False,
            "einet_reactivation_type": "param_tanh"
        },
        "MLP": {
            "net_type": "MLP"
        }
    }
    
    results = run_scaling_experiment(BASE_CONFIG, DIM_LIST, experiment_configs, EPOCHS)
    
    plot_fig = plot_scaling_laws(results)
    plt.show()

Files already downloaded and verified
Files already downloaded and verified


2025-02-25 13:28:34,286 - scaling_notebook - INFO - Creating EINet network
2025-02-25 13:28:34,289 - scaling_notebook - INFO - EINet config={'net_type': 'EINet', 'einet_use_shunting': True, 'einet_reactivation_type': 'param_tanh', 'dim': 2}: Parameter count = 72210


RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x3072 and 784x20)